# Full-range TURN / RIVER pass, headless GPU

Upgrades the turn/river prototype from a **reduced range** (N=90) to the **full range**
(N=400) across the 16 curated runouts (8 turn + 8 river — bricks, flush-completers,
board-pairers, overcards). These are 1–2 street solves (much cheaper than the flop's
3-street), so the whole thing runs in **one commit** (~20–40 min).

**One commit:** Accelerator → GPU T4 x2, Internet On, then Save & Run All. Download
`flop_pack_turnriver_fullrange.db` (+ `.gz`) and send them back — they drop straight into
the trainer, replacing the demo turn/river pack.

*(Still unconditioned — ranges are the pre-flop SRP with card-removal only, NOT filtered
by a prior check/check line. The trainer copy already says so.)*

In [ ]:
# Fail fast if no GPU.
import subprocess
try:
    _r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit('No nvidia-smi (CPU instance) -- set Accelerator -> GPU, then re-run.')
assert _r.returncode == 0 and 'GPU' in _r.stdout, 'NO GPU -- set Accelerator -> GPU T4 x2, then re-run.'
print(_r.stdout.strip())

In [ ]:
# Clone the solver source (needs Internet On).
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
print('source ready')

In [ ]:
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
free,total=cp.cuda.Device(0).mem_info
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; print('cupy',cp.__version__,'|',nm.decode() if isinstance(nm,bytes) else nm)
print('GPU memory: %.1f GB free / %.1f GB total'%(free/1e9,total/1e9))

In [ ]:
# Solve all 16 runouts at full range on GPU (~20-40 min).
import subprocess, os
env = {**os.environ, 'PYTHONPATH': 'src'}
subprocess.run(
    ['python', 'demo/gen_turn_river.py',
     '--solver', 'gpu', '--dtype', 'float32', '--n', '400', '--iters', '300',
     '--version', 'turnriver_fullrange',
     '--note', 'turn/river full range (NOT check-check filtered)'],
    cwd='/kaggle/working/poker', env=env, check=True)

In [ ]:
# Expose the pack for download.
import shutil, os
src='/kaggle/working/poker/output/packs/flop_pack_turnriver_fullrange.db'
if not os.path.exists(src):
    raise SystemExit('No turnriver_fullrange pack -- check the solve cell above.')
shutil.copy(src, '/kaggle/working/flop_pack_turnriver_fullrange.db')
shutil.copy(src+'.gz', '/kaggle/working/flop_pack_turnriver_fullrange.db.gz')
print('DOWNLOAD -> flop_pack_turnriver_fullrange.db (%d KB) + .gz'
      % (os.path.getsize('/kaggle/working/flop_pack_turnriver_fullrange.db')//1024))